# Feature Hashing & Feature Generation

This notebook covers two powerful feature engineering techniques:

| Technique | Description |
|---|---|
| **Feature Hashing** | Maps high-cardinality categorical values into a fixed-size numeric vector using a hash function — no vocabulary needed |
| **Feature Generation** | Creates new informative features from existing ones via ratios, multiplicative interactions, and polynomial combinations |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction import FeatureHasher
from sklearn.preprocessing import PolynomialFeatures

sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Train.csv")

print("Dataset Shape:", df.shape)
df.head()

## 3. Data Overview

In [ ]:
print("Column\n", df.columns)
print("Missing Value\n", df.isnull().sum())

## 4. Feature Hashing

Feature hashing (a.k.a. the *hashing trick*) encodes categorical values into a fixed-size numeric array without needing to maintain a category vocabulary. Useful for high-cardinality features or online learning.

### 4.1 Single Column — `Mode_of_Shipment` (4 hash buckets)

In [ ]:
mode_list = df['Mode_of_Shipment'].astype(str).apply(lambda x: [x])

hasher = FeatureHasher(n_features=4, input_type='string')
hashed_features = hasher.transform(mode_list)

hashed_df = pd.DataFrame(
    hashed_features.toarray(),
    columns=[f"Mode_hash_{i}" for i in range(4)]
)

print("Hashed Features Shape:", hashed_df.shape)
hashed_df.head()

### 4.2 Multi-Column — `Mode_of_Shipment` + `Warehouse_block` (6 hash buckets)

In [ ]:
cat_cols = ['Mode_of_Shipment', 'Warehouse_block']
combined = df[cat_cols].astype(str).agg(' '.join, axis=1)
combined_tokens = combined.apply(lambda x: x.split())

hasher_multi = FeatureHasher(n_features=6, input_type='string')
hashed_multi = hasher_multi.transform(combined_tokens)

hashed_multi_df = pd.DataFrame(
    hashed_multi.toarray(),
    columns=[f"Hash_{i}" for i in range(6)]
)

print("Hashed Multi Shape:", hashed_multi_df.shape)
hashed_multi_df.head()

## 5. Feature Generation

Manual feature engineering — creating new columns that express domain-meaningful relationships.

### 5.1 Cost per Gram

> Captures the unit cost relative to product weight. `+1` prevents division by zero.

In [ ]:
df['Cost_per_Gram'] = df['Cost_of_the_Product'] / (df['Weight_in_gms'] + 1)

df[['Cost_of_the_Product', 'Weight_in_gms', 'Cost_per_Gram']].head()

### 5.2 Discount Ratio

> The fraction of the product cost that is discounted. `+1` prevents division by zero.

In [ ]:
df['Discount_Ratio'] = df['Discount_offered'] / (df['Cost_of_the_Product'] + 1)

df[['Discount_offered', 'Cost_of_the_Product', 'Discount_Ratio']].head()

### 5.3 Customer Engagement Score

> Product of `Customer_care_calls` × `Customer_rating` — a proxy for how actively a customer interacted with support.

In [ ]:
df['Customer_Engagement'] = df['Customer_care_calls'] * df['Customer_rating']

df[['Customer_care_calls', 'Customer_rating', 'Customer_Engagement']].head()

### 5.4 Polynomial Interaction Features

Uses `PolynomialFeatures(degree=2, interaction_only=True)` to generate the pairwise product of `Cost_of_the_Product` and `Discount_offered`, capturing their joint effect without including squared terms.

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=True)

cols = ['Cost_of_the_Product', 'Discount_offered']
poly_features = poly.fit_transform(df[cols])
poly_feature_names = poly.get_feature_names_out(cols)

poly_df = pd.DataFrame(poly_features, columns=poly_feature_names)
poly_df.head()

## 6. Combine All Features

In [ ]:
df_final = pd.concat([df, hashed_df, hashed_multi_df, poly_df], axis=1)

print("Final Dataset Shape:", df_final.shape)
df_final.head()